In [2]:
import tensorflow as tf
import numpy as np
import sklearn
from sklearn import metrics
import json
import matplotlib.pyplot as plt
import random
import seaborn as sn
import pandas as pd
# load base package for the tasks from pytorch
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [5]:
df = pd.read_csv(r"C:\Users\adiks\OneDrive\Documents\GitHub\Detecting-and-Classifying-Hate-Speech-in-online-tweets\processed_tweets.csv")

In [7]:
df.cyberbullying_type.value_counts()
df = df.dropna(axis=0)
df.describe().T


,count,unique,top,freq
tweet_text,47483,45395,rt:,17
cyberbullying_type,47483,6,religion,7998
tokenized_tweet,47483,45336,"['rt', ':']",20


In [8]:
X = df['tweet_text']
y = df['cyberbullying_type']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=40, stratify=y)



In [9]:
sentences = X_train.values
labels = y_train.values

In [12]:
from transformers import RobertaTokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

# Print the original sentence.
print(' Original: ', sentences[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(sentences[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(sentences[0])))


 Original:  female: you cant call me a bitch! her bio: im that bitch me:
Tokenized:  ['female', ':', 'Ġyou', 'Ġcant', 'Ġcall', 'Ġme', 'Ġa', 'Ġbitch', '!', 'Ġher', 'Ġbio', ':', 'Ġim', 'Ġthat', 'Ġbitch', 'Ġme', ':']
Token IDs:  [31372, 35, 47, 17672, 486, 162, 10, 32594, 328, 69, 10709, 35, 4356, 14, 32594, 162, 35]


In [13]:
# Find maximum length based on our dataset
max_len = 0

# For every sentence...
for sent in sentences:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(sent, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max sentence length: ', max_len)

Token indices sequence length is longer than the specified maximum sequence length for this model (689 > 512). Running this sequence through the model will result in indexing errors


Max sentence length:  1083


In [14]:
# Create empty lists to store outputs
input_ids = []
attention_masks = []

# For every sentence...
for sent in sentences:
    # `encode_plus` will:
    #    (1) Tokenize the sentence
    #    (2) Add the `[CLS]` and `[SEP]` token to the start and end
    #    (3) Truncate/Pad sentence to max length
    #    (4) Map tokens to their IDs
    #    (5) Create attention mask
    #    (6) Return a dictionary of outputs
    encoded_sent = tokenizer.encode_plus(
        text=sent,  # Preprocess sentence
        add_special_tokens=True,        # Add `[CLS]` and `[SEP]`
        max_length=150,             # Max length to truncate/pad
        pad_to_max_length=True,         # Pad sentence to max length
        return_attention_mask=True,      # Return attention mask
        truncation = True
        )
    # Add the outputs to the lists
    input_ids.append(encoded_sent.get('input_ids'))
    attention_masks.append(encoded_sent.get('attention_mask'))
# Convert lists to tensors
input_ids = torch.tensor(input_ids)
attention_masks = torch.tensor(attention_masks)
labels = torch.tensor(labels)


c:\Users\adiks\OneDrive\Documents\GitHub\Detecting-and-Classifying-Hate-Speech-in-online-tweets\cmpe257\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [ ]:
# Print sentence 0, now as a list of IDs.
print('Original: ', sentences[0])
print('Token IDs:', input_ids[0])